# DNA Sequence ML Analysis
**Goal:** Fetch real gene sequences from NCBI, extract biologically meaningful features, and train a Random Forest classifier to predict gene type.

**Pipeline:**
1. Fetch sequences from NCBI Entrez (protein-coding genes, rRNA, tRNA)
2. Extract features: GC content, k-mer frequencies, CpG ratio, homopolymer runs
3. Train + evaluate a Random Forest classifier
4. Visualise confusion matrix and feature importances

In [ ]:
# ── 0. Imports ────────────────────────────────────────────────────────────────
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from Bio import Entrez, SeqIO
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from dna_features import sequences_to_dataframe

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('All imports OK')

In [ ]:
# ── 1. NCBI Configuration ─────────────────────────────────────────────────────
Entrez.email = "farhanabhuiyanliza@gmail.com"   # required by NCBI
# Entrez.api_key = "YOUR_API_KEY_HERE"  # uncomment if you have an NCBI API key

GENE_CLASSES = {
    "protein_coding": '"mRNA"[Feature Key] AND Homo sapiens[Organism] AND 500:3000[Sequence Length]',
    "rRNA":           '"rRNA"[Feature Key] AND Homo sapiens[Organism] AND 100:2000[Sequence Length]',
    "tRNA":           '"tRNA"[Feature Key] AND Homo sapiens[Organism] AND 60:200[Sequence Length]',
}
MAX_PER_CLASS = 150   # fetch up to 150 per class (450 total); raise for more data
print(f'Fetching up to {MAX_PER_CLASS} sequences per class from NCBI...')

In [ ]:
# ── 2. Fetch sequences from NCBI ──────────────────────────────────────────────
def fetch_sequences(query: str, label: str, max_records: int = 150) -> list:
    """Search NCBI nucleotide DB and return list of record dicts."""
    records = []
    try:
        # Search
        handle = Entrez.esearch(db="nucleotide", term=query, retmax=max_records)
        search_results = Entrez.read(handle)
        handle.close()
        id_list = search_results["IdList"]
        print(f"  [{label}] found {len(id_list)} IDs")

        if not id_list:
            return records

        # Fetch in batches of 50 to avoid NCBI timeouts
        batch_size = 50
        for start in range(0, len(id_list), batch_size):
            batch = id_list[start : start + batch_size]
            fetch_handle = Entrez.efetch(
                db="nucleotide",
                id=",".join(batch),
                rettype="fasta",
                retmode="text",
            )
            for seq_rec in SeqIO.parse(fetch_handle, "fasta"):
                seq_str = str(seq_rec.seq).upper()
                # Skip sequences with ambiguous bases > 5 %
                valid = sum(seq_str.count(n) for n in "ATGC")
                if valid / max(len(seq_str), 1) < 0.95:
                    continue
                records.append({
                    "id":          seq_rec.id,
                    "description": seq_rec.description,
                    "sequence":    seq_str,
                    "label":       label,
                })
            fetch_handle.close()
            time.sleep(0.35)   # respect NCBI rate limit (3 req/s without key)

        print(f"  [{label}] kept {len(records)} clean sequences")
    except Exception as exc:
        print(f"  [{label}] ERROR: {exc}")
    return records


all_records = []
for label, query in GENE_CLASSES.items():
    recs = fetch_sequences(query, label, max_records=MAX_PER_CLASS)
    all_records.extend(recs)

print(f"\nTotal sequences collected: {len(all_records)}")

In [ ]:
# ── 3. Feature Extraction ─────────────────────────────────────────────────────
df = sequences_to_dataframe(all_records)
print(f"Feature matrix shape: {df.shape}")
print(f"\nClass distribution:")
print(df['label'].value_counts())
df.head(3)

In [ ]:
# ── 4. Exploratory Data Analysis ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, feat in zip(axes, ["gc_content", "cpg_ratio", "length"]):
    for label, grp in df.groupby("label"):
        ax.hist(grp[feat], bins=30, alpha=0.6, label=label, density=True)
    ax.set_title(feat.replace("_", " ").title())
    ax.set_xlabel(feat)
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.suptitle("Feature Distributions by Gene Class", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("plots/eda_distributions.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5. Prepare Data for ML ────────────────────────────────────────────────────
feature_cols = [c for c in df.columns if c != "label"]
X = df[feature_cols].values.astype(float)

le = LabelEncoder()
y = le.fit_transform(df["label"])
class_names = le.classes_

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")
print(f"Classes: {class_names}")

In [ ]:
# ── 6. Train Random Forest ────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

# 5-fold stratified cross-validation on full dataset
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X, y, cv=cv, scoring="f1_macro", n_jobs=-1)
print(f"CV F1-macro:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# ── 7. Evaluation ─────────────────────────────────────────────────────────────
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=class_names))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.savefig("plots/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8. Feature Importance ─────────────────────────────────────────────────────
importances = rf.feature_importances_
feat_df = pd.DataFrame({"feature": feature_cols, "importance": importances})
feat_df = feat_df.sort_values("importance", ascending=False).head(25)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(
    data=feat_df,
    x="importance",
    y="feature",
    hue="feature",
    palette="viridis",
    legend=False,
    ax=ax,
)
ax.set_title("Top-25 Feature Importances (Random Forest)", fontsize=13)
ax.set_xlabel("Mean Decrease in Impurity")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("plots/feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()

feat_df.head(10)

In [ ]:
# ── 9. GC Content Correlation Heatmap (top features) ─────────────────────────
top_feats = feat_df["feature"].tolist()
corr = df[top_feats].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr,
    ax=ax,
    cmap="coolwarm",
    center=0,
    linewidths=0.3,
    annot=False,
)
ax.set_title("Feature Correlation Heatmap (Top-25 Features)", fontsize=13)
plt.tight_layout()
plt.savefig("plots/correlation_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 10. Save Feature Matrix ───────────────────────────────────────────────────
df.to_csv("data/feature_matrix.csv")
print("Feature matrix saved to data/feature_matrix.csv")

## Results Summary

| Metric | Value |
|--------|-------|
| Model | Random Forest (300 trees) |
| Features | GC content, AT/GC ratio, CpG ratio, mono/di/tri-nucleotide freq., homopolymer runs |
| Classes | protein_coding, rRNA, tRNA |
| CV F1-macro | see output above |

**Key findings:**
- tRNA sequences are short and have a distinct nucleotide composition, making them the easiest class to separate.
- CpG ratio and GC content are among the most discriminative features.
- Di- and tri-nucleotide frequencies (k-mers) capture codon-usage biases that help distinguish protein-coding from non-coding RNA.